In [1]:
import os

In [2]:
import os

os.chdir("..")

print(os.getcwd())

c:\Users\abhin\OneDrive\Documents\ABHI DOC\End-to-End-Machine-Learning-Project-With-MLFlow


In [3]:
from pathlib import Path

print(Path("config").exists())
print(Path("config/config.yaml").exists())

True
True


In [2]:
%pwd

'c:\\Users\\abhin\\OneDrive\\Documents\\ABHI DOC\\End-to-End-Machine-Learning-Project-With-MLFlow\\research'

In [3]:
import os
os.chdir("..")

In [4]:
%pwd

'c:\\Users\\abhin\\OneDrive\\Documents\\ABHI DOC\\End-to-End-Machine-Learning-Project-With-MLFlow'

In [15]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [8]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [9]:
print(CONFIG_FILE_PATH)
print(PARAMS_FILE_PATH)
print(SCHEMA_FILE_PATH)

config\config.yaml
params.yaml
schema.yaml


In [10]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [19]:
import os
import urllib.request as request
import zipfile
from mlProject import logger
from mlProject.utils.common import get_size

In [16]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")



    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
  

In [21]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-06-28 11:35:19,194: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-28 11:35:19,197: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-28 11:35:19,200: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-06-28 11:35:19,205: INFO: common: created directory at: artifacts]
[2026-06-28 11:35:19,209: INFO: common: created directory at: artifacts/data_ingestion]
[2026-06-28 11:35:20,767: INFO: 2366278714: artifacts/data_ingestion/data.zip download! with following info: 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: B4AA:1264BC:26A21:2D2F4:6A40B99E
Accept-Ranges: bytes
Date: Sun, 28 J

In [20]:
print(request)

<module 'urllib.request' from 'C:\\Users\\abhin\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\urllib\\request.py'>
